# PRESENTATION RUN GUIDE

**Open:** `notebooks/03_transfer_learning/03_transfer_learning_comparison.ipynb`. Select the project Python kernel installed from `requirements.txt`, and launch Jupyter from the repository root.

**Before training:** set `SPLIT_DIR` in Section 3 if needed and verify the prepared data against `data/dataset_manifest.csv`. Section 4 repeats this check before fitting anything.

**Run order:** execute top to bottom. Sections 1?8 configure the data and build the models. Section 9 trains and saves two frozen-backbone models. Section 10 loads those checkpoints, unfreezes only the final configured layers, and trains two separate fine-tuned models. Sections 11?15 create histories and evaluate all four configurations. Section 16 reload-verifies the complete checkpoints.

> **Run All performs four training phases** (DenseNet121 frozen and fine-tuned; ResNet50 frozen and fine-tuned). Do not use Run All for a presentation-only walkthrough. No training was run as part of this reorganization.

**Presentation summary:** We compare ImageNet-pretrained DenseNet121 and ResNet50 on a fixed pediatric X-ray split. First, only the new classifier head is trained while the complete backbone is frozen. Then a copy of each resulting frozen checkpoint is fine-tuned by unfreezing only its final 30 backbone layers at a lower learning rate. Validation data controls early stopping and threshold selection; the held-out test set is used only for final evaluation.


## 1. Project Objective

Transfer learning starts from visual features learned on ImageNet, then adapts them to NORMAL/PNEUMONIA classification. DenseNet121 reuses features through dense connections; ResNet50 uses residual skip connections. Both use the same verified split, 224?224 RGB inputs, an architecture-specific preprocessing function, training-only augmentation, and training-only class weights. This academic prototype is not a diagnostic tool.


## 2. Imports

TensorFlow/Keras supplies the data pipeline and models. Scikit-learn computes test metrics after thresholds have been selected on validation data.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


## 3. Configuration

All model and result paths are relative to the project root. Set `SPLIT_DIR` to the prepared folder if automatic KaggleHub-cache discovery is not appropriate. The fine-tuning phase starts from the frozen checkpoint made in this run.


In [ ]:
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from this project or set PROJECT_ROOT explicitly.")
sys.path.insert(0, str(PROJECT_ROOT))
from src.evaluation.binary_metrics import save_training_history

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
RUN_ID = "seed42_run01"
TARGET_SENSITIVITY = 0.90
HEAD_EPOCHS = 5
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LAYERS = 30
FINE_TUNE_LEARNING_RATE = 1e-5
AUTOTUNE = tf.data.AUTOTUNE
MANIFEST_SUMMARY = json.loads((PROJECT_ROOT / "data" / "dataset_manifest_summary.json").read_text(encoding="utf-8"))
MANIFEST_SHA256 = MANIFEST_SUMMARY["manifest_sha256"]

SPLIT_DIR = None  # Example: Path(r"C:/path/to/chest_xray_cnn_splits")
FROZEN_DIR = PROJECT_ROOT / "models" / "frozen"
FINE_TUNED_DIR = PROJECT_ROOT / "models" / "fine_tuned"
RESULTS_DIR = PROJECT_ROOT / "results" / "transfer_learning" / RUN_ID
if RESULTS_DIR.exists() and any(path.is_file() for path in RESULTS_DIR.rglob("*")):
    raise FileExistsError(
        f"Refusing to overwrite results under {RESULTS_DIR}. Change RUN_ID in Section 3."
    )
RESULTS_FOLDERS = {
    "metrics": RESULTS_DIR / "metrics",
    "history": RESULTS_DIR / "training_history",
    "confusion": RESULTS_DIR / "confusion_matrices",
    "roc": RESULTS_DIR / "roc_curves",
    "pr": RESULTS_DIR / "precision_recall",
    "comparison": RESULTS_DIR / "model_comparison",
}
for folder in RESULTS_FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)

FROZEN_DIR.mkdir(parents=True, exist_ok=True)
FINE_TUNED_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print("Project:", PROJECT_ROOT)
print("Frozen models:", FROZEN_DIR)
print("Fine-tuned models:", FINE_TUNED_DIR)
print("Results:", RESULTS_DIR)


## 4. Dataset Loading

The class folders define the labels. Training shuffles; validation and test preserve deterministic file order. The manifest checker verifies each file path and SHA-256 before any model is trained.


In [ ]:
if SPLIT_DIR is None:
    cache_versions = (
        Path.home() / ".cache" / "kagglehub" / "datasets" /
        "paultimothymooney" / "chest-xray-pneumonia" / "versions"
    )
    candidates = [
        PROJECT_ROOT / "data" / "chest_xray_cnn_splits",
        PROJECT_ROOT / "chest_xray_cnn_splits",
        cache_versions / "chest_xray_cnn_splits",
        *sorted(cache_versions.glob("*/chest_xray_cnn_splits")),
    ]
    SPLIT_DIR = next(
        (path for path in candidates if all((path / split).is_dir() for split in ("train", "val", "test"))),
        None,
    )
if SPLIT_DIR is None or not all((Path(SPLIT_DIR) / split).is_dir() for split in ("train", "val", "test")):
    raise FileNotFoundError("Set SPLIT_DIR to the prepared train/val/test folder matching data/dataset_manifest.csv.")
SPLIT_DIR = Path(SPLIT_DIR).resolve()

def verify_manifest():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "src" / "data" / "validate_dataset.py"), "--split-dir", str(SPLIT_DIR)],
        check=True,
    )


verify_manifest()

train_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "train", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    seed=SEED, label_mode="int", color_mode="rgb",
)
class_names = train_ds.class_names
if class_names != ["NORMAL", "PNEUMONIA"]:
    raise ValueError(f"Expected class order ['NORMAL', 'PNEUMONIA']; found {class_names}")
val_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DIR / "val", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    shuffle=False, class_names=class_names, label_mode="int", color_mode="rgb",
)
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
print("Data:", SPLIT_DIR, "| class mapping:", {name: index for index, name in enumerate(class_names)})


## 5. Preprocessing and Augmentation

The directory loader resizes images to 224?224 RGB. Each model applies its matching Keras `preprocess_input` layer. Random flip, small rotation, and zoom are embedded in the model and are active during training only; validation and test inputs are unaugmented.


In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.02, seed=SEED + 1),
        tf.keras.layers.RandomZoom(0.05, seed=SEED + 2),
    ],
    name="training_augmentation",
)


## 6. Class Distribution and Class Weights

Inverse-frequency class weights are calculated from the training split only. Validation and test distributions are left unchanged.


In [ ]:
train_counts = np.zeros(len(class_names), dtype=np.int64)
for _, labels in train_ds:
    train_counts += np.bincount(labels.numpy().reshape(-1), minlength=len(class_names))
if np.any(train_counts == 0):
    raise ValueError(f"A training class is absent: {dict(zip(class_names, train_counts.tolist()))}")
class_weight = {
    class_id: float(train_counts.sum() / (len(class_names) * count))
    for class_id, count in enumerate(train_counts)
}
print("Training counts:", dict(zip(class_names, train_counts.tolist())))
print("Class weights:", class_weight)


## 7. DenseNet121 Architecture

DenseNet121 uses ImageNet weights without the original ImageNet classifier. Global average pooling, dropout, and a two-class softmax form the new head. The full backbone begins frozen.


In [ ]:
def make_model(architecture):
    if architecture == "DenseNet121":
        backbone = tf.keras.applications.DenseNet121(
            include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3)
        )
        preprocess = tf.keras.applications.densenet.preprocess_input
    elif architecture == "ResNet50":
        backbone = tf.keras.applications.ResNet50(
            include_top=False, weights="imagenet", input_shape=(*IMAGE_SIZE, 3)
        )
        preprocess = tf.keras.applications.resnet50.preprocess_input
    else:
        raise ValueError(f"Unsupported architecture: {architecture}")

    backbone.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = augmentation(inputs)
    x = tf.keras.layers.Lambda(preprocess, name="preprocess")(x)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name=architecture.lower())
    return model, backbone

densenet_model, densenet_backbone = make_model("DenseNet121")
if densenet_model.input_shape != (None, 224, 224, 3) or densenet_model.output_shape != (None, 2):
    raise ValueError("DenseNet121 model has an unexpected input/output shape.")
print("DenseNet121:", densenet_model.input_shape, "->", densenet_model.output_shape)


## 8. ResNet50 Architecture

ResNet50 uses residual skip connections. It uses the same classifier-head design, ImageNet initialization, and 224?224 inputs, with ResNet-specific preprocessing.


In [ ]:
resnet_model, resnet_backbone = make_model("ResNet50")
if resnet_model.input_shape != (None, 224, 224, 3) or resnet_model.output_shape != (None, 2):
    raise ValueError("ResNet50 model has an unexpected input/output shape.")
print("ResNet50:", resnet_model.input_shape, "->", resnet_model.output_shape)

frozen_models = {
    "DenseNet121": (densenet_model, densenet_backbone),
    "ResNet50": (resnet_model, resnet_backbone),
}
preprocess_functions = {
    "DenseNet121": tf.keras.applications.densenet.preprocess_input,
    "ResNet50": tf.keras.applications.resnet50.preprocess_input,
}


## 9. Frozen Backbone Training

Each ImageNet backbone stays frozen while its classification head trains for up to five epochs. Early stopping monitors validation loss; the existing ReduceLROnPlateau callback is retained. The complete frozen checkpoint is saved before fine-tuning begins.


In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=1, min_lr=1e-7
        ),
    ]

def verify_frozen_backbone(name, backbone):
    assert backbone.trainable is False, f"{name}: backbone is not frozen."
    assert len(backbone.trainable_weights) == 0, f"{name}: backbone has trainable weights."

head_histories = {}
frozen_paths = {
    name: FROZEN_DIR / f"{name}_{RUN_ID}_frozen.keras"
    for name in frozen_models
}
fine_tuned_paths = {
    name: FINE_TUNED_DIR / f"{name}_{RUN_ID}_fine_tuned.keras"
    for name in frozen_models
}
existing_checkpoints = [path for path in [*frozen_paths.values(), *fine_tuned_paths.values()] if path.exists()]
if existing_checkpoints:
    raise FileExistsError(
        "Refusing to overwrite existing checkpoints. Change RUN_ID in Section 3: "
        + ", ".join(str(path) for path in existing_checkpoints)
    )
for name, (model, backbone) in frozen_models.items():
    verify_manifest()
    verify_frozen_backbone(name, backbone)
    print(f"\nTraining {name} - Frozen Backbone")
    check_frozen = tf.keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs, b=backbone, n=name: verify_frozen_backbone(n, b)
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(HEAD_LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    head_histories[name] = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=class_weight,
        callbacks=[*make_callbacks(), check_frozen],
        verbose=2,
    )
    save_training_history(head_histories[name], RESULTS_FOLDERS["history"] / f"{name}_frozen_history.csv")
    frozen_config = {
        "model": name, "run_id": RUN_ID, "strategy": "frozen",
        "seed": SEED, "manifest_sha256": MANIFEST_SHA256,
        "class_mapping": {class_name: index for index, class_name in enumerate(class_names)},
        "training_counts": dict(zip(class_names, train_counts.tolist())),
        "class_weights": {str(key): float(value) for key, value in class_weight.items()},
        "image_size": list(IMAGE_SIZE), "batch_size": BATCH_SIZE,
        "max_epochs": HEAD_EPOCHS, "learning_rate": HEAD_LEARNING_RATE,
        "optimizer": "Adam", "loss": "sparse_categorical_crossentropy",
        "checkpoint": str(frozen_paths[name].resolve()),
        "augmentation": ["horizontal_flip", "rotation_0.02", "zoom_0.05"],
    }
    (RESULTS_DIR / f"{name}_frozen_config.json").write_text(json.dumps(frozen_config, indent=2) + "\n", encoding="utf-8")
    verify_frozen_backbone(name, backbone)
    head_parameters = sum(int(np.prod(weight.shape)) for weight in model.trainable_weights)
    print(
        f"{name}: backbone={backbone.name}; trainable={backbone.trainable}; "
        f"trainable backbone weights={len(backbone.trainable_weights)}; "
        f"trainable head parameters={head_parameters:,}"
    )
    model.save(frozen_paths[name])
    loaded_frozen = tf.keras.models.load_model(
        frozen_paths[name], compile=False, safe_mode=True,
        custom_objects={"preprocess_input": preprocess_functions[name]},
    )
    verify_frozen_backbone(name, loaded_frozen.get_layer(name.lower()))
    print("Saved and reloaded frozen model:", frozen_paths[name])


## 10. Fine-Tuning

For each architecture, load the complete frozen checkpoint just saved. Unfreeze only its final 30 backbone layers; earlier layers stay frozen. Recompile with Adam at `1e-5`, a lower learning rate than the head-only phase, and train for up to ten epochs. These models and histories are separate from the frozen models.


In [ ]:
fine_tuned_models = {}
fine_tune_histories = {}
for name in frozen_models:
    verify_manifest()
    model = tf.keras.models.load_model(
        frozen_paths[name],
        compile=False,
        safe_mode=True,
        custom_objects={"preprocess_input": preprocess_functions[name]},
    )
    backbone = model.get_layer(name.lower())
    if backbone.trainable:
        raise ValueError(f"{name}: loaded frozen checkpoint has a trainable backbone.")
    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False
    trainable_layers = sum(layer.trainable for layer in backbone.layers)
    print(f"{name}: fine-tuning {trainable_layers} of {len(backbone.layers)} backbone layers")
    if trainable_layers != min(FINE_TUNE_LAYERS, len(backbone.layers)):
        raise ValueError(f"{name}: unexpected number of trainable backbone layers.")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(FINE_TUNE_LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    fine_tune_histories[name] = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weight,
        callbacks=make_callbacks(),
        verbose=2,
    )
    fine_tuned_models[name] = model
    save_training_history(fine_tune_histories[name], RESULTS_FOLDERS["history"] / f"{name}_fine_tuned_history.csv")
    fine_tuned_config = {
        "model": name, "run_id": RUN_ID, "strategy": "fine_tuned",
        "seed": SEED, "manifest_sha256": MANIFEST_SHA256,
        "class_mapping": {class_name: index for index, class_name in enumerate(class_names)},
        "training_counts": dict(zip(class_names, train_counts.tolist())),
        "class_weights": {str(key): float(value) for key, value in class_weight.items()},
        "image_size": list(IMAGE_SIZE), "batch_size": BATCH_SIZE,
        "max_epochs": FINE_TUNE_EPOCHS, "learning_rate": FINE_TUNE_LEARNING_RATE,
        "optimizer": "Adam", "loss": "sparse_categorical_crossentropy",
        "fine_tune_layers": FINE_TUNE_LAYERS,
        "initialized_from": str(frozen_paths[name].resolve()),
        "checkpoint": str(fine_tuned_paths[name].resolve()),
        "augmentation": ["horizontal_flip", "rotation_0.02", "zoom_0.05"],
    }
    (RESULTS_DIR / f"{name}_fine_tuned_config.json").write_text(json.dumps(fine_tuned_config, indent=2) + "\n", encoding="utf-8")
    model.save(fine_tuned_paths[name])
    loaded_fine_tuned = tf.keras.models.load_model(
        fine_tuned_paths[name], compile=False, safe_mode=True,
        custom_objects={"preprocess_input": preprocess_functions[name]},
    )
    if not loaded_fine_tuned.get_layer(name.lower()).trainable:
        raise ValueError(f"{name}: saved fine-tuned backbone is not trainable.")
    print("Saved and reloaded fine-tuned model:", fine_tuned_paths[name])


## 11. Training History Comparison

The plots compare training and validation accuracy and loss for the head-only and fine-tuning phases. Each phase has its own epoch axis because fine-tuning starts from the best frozen checkpoint.


In [ ]:
for name in frozen_models:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for metric, axis in (("accuracy", axes[0]), ("loss", axes[1])):
        for phase, history in (("Frozen", head_histories[name]), ("Fine-tuned", fine_tune_histories[name])):
            axis.plot(history.history[metric], label=f"{phase} training")
            axis.plot(history.history[f"val_{metric}"], linestyle="--", label=f"{phase} validation")
        axis.set(title=f"{name}: {metric}", xlabel="Epoch", ylabel=metric.capitalize())
        axis.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_FOLDERS["history"] / f"{name}_training_history.png", dpi=140)
    display(fig)
    plt.close(fig)


## 12. Test Evaluation

For each of four models, choose the PNEUMONIA threshold that maximizes F1 on validation data. Fix that threshold before evaluating test predictions. Keras test loss and argmax accuracy are also reported; the confusion matrix and thresholded metrics use the validation-selected threshold.


In [ ]:
evaluation_models = {
    "DenseNet121_Frozen": frozen_models["DenseNet121"][0],
    "DenseNet121_FineTuned": fine_tuned_models["DenseNet121"],
    "ResNet50_Frozen": frozen_models["ResNet50"][0],
    "ResNet50_FineTuned": fine_tuned_models["ResNet50"],
}
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "src" / "evaluation" / "evaluate_experiments.py"),
        "--run-id", RUN_ID,
        "--split-dir", str(SPLIT_DIR),
    ],
    check=True,
)
selection_path = PROJECT_ROOT / "results" / "metrics" / f"model_selection_{RUN_ID}.json"
selection_record = json.loads(selection_path.read_text(encoding="utf-8"))
comparison_csv = PROJECT_ROOT / "results" / "metrics" / f"model_comparison_{RUN_ID}.csv"
with comparison_csv.open(newline="", encoding="utf-8") as stream:
    comparison_rows = list(csv.DictReader(stream))
evaluation_results = {row["model"]: row for row in comparison_rows}
print("Validation-only selected model:", selection_record["selected_model"])
print("Selection rule:", selection_record["selection_rule"])
print("Seven-model held-out test results:", comparison_csv)
for name, row in evaluation_results.items():
    print(
        f"{name}: threshold={float(row['threshold']):.4f}, "
        f"loss={float(row['test_loss']):.4f}, accuracy={float(row['accuracy']):.4f}, "
        f"sensitivity={float(row['sensitivity_recall']):.4f}, "
        f"specificity={float(row['specificity']):.4f}, F1={float(row['f1_score']):.4f}"
    )


## 13. Confusion Matrices

Rows are true labels and columns are predicted labels. These matrices use each model's threshold selected using validation data.


In [ ]:
from IPython.display import Image

for name in (
    "DenseNet121_Frozen", "DenseNet121_FineTuned",
    "ResNet50_Frozen", "ResNet50_FineTuned",
):
    matrix_path = RESULTS_DIR / "per_model" / name / "confusion_matrix.png"
    display(Image(filename=str(matrix_path)))


## 14. ROC and Precision-Recall Analysis

ROC-AUC and average precision are computed from test probabilities, not from thresholds. The test set remains a final reporting set and is not used to select thresholds.


In [ ]:
from IPython.display import Image

for name in (
    "DenseNet121_Frozen", "DenseNet121_FineTuned",
    "ResNet50_Frozen", "ResNet50_FineTuned",
):
    for filename in ("roc_curve.png", "precision_recall_curve.png"):
        display(Image(filename=str(RESULTS_DIR / "per_model" / name / filename)))


## 15. Model Comparison

This comparison uses the validation-selected threshold metrics for each of the four configurations. Preserve the CSV with the split manifest hash when reporting results.


In [ ]:
from IPython.display import Markdown

comparison_markdown = PROJECT_ROOT / "results" / "metrics" / f"model_comparison_{RUN_ID}.md"
display(Markdown(comparison_markdown.read_text(encoding="utf-8")))
print("Unified comparison plot:", PROJECT_ROOT / "results" / "plots" / f"model_comparison_{RUN_ID}.png")


## 16. Saving Models

Save complete `.keras` models, not weights alone, to separate frozen and fine-tuned folders. Reload each with its architecture-specific preprocessing function and verify the expected input/output shapes.


In [ ]:
model_paths = {
    **{f"{name}_Frozen": frozen_paths[name] for name in frozen_models},
    **{f"{name}_FineTuned": fine_tuned_paths[name] for name in frozen_models},
}
for name, path in model_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Expected complete checkpoint is missing: {path}")
    architecture = "DenseNet121" if name.startswith("DenseNet121") else "ResNet50"
    reloaded = tf.keras.models.load_model(
        path,
        compile=False,
        safe_mode=True,
        custom_objects={"preprocess_input": preprocess_functions[architecture]},
    )
    if reloaded.input_shape != (None, 224, 224, 3) or reloaded.output_shape != (None, 2):
        raise ValueError(f"Saved {name} checkpoint has incompatible shapes.")
    backbone = reloaded.get_layer(architecture.lower())
    if name.endswith("_Frozen") and (backbone.trainable or backbone.trainable_weights):
        raise ValueError(f"{name}: saved backbone is not frozen.")
    print(f"Verified complete model: {path.resolve()} ({path.stat().st_size:,} bytes)")


## 17. Final Discussion and Conclusion

Report the four-model metrics together with the verified dataset manifest. Compare per-class sensitivity and specificity, not accuracy alone. The older baseline metrics are historical and use an unmanifested train/validation allocation. Explain that this dataset has no patient-level split guarantee and that neither test metrics nor Grad-CAM establish clinical validity.


In [ ]:
print("Validation-only selected model:", selection_record["selected_model"])
for name, row in evaluation_results.items():
    print(
        f"{name}: accuracy={float(row['accuracy']):.4f}, precision={float(row['precision']):.4f}, "
        f"sensitivity={float(row['sensitivity_recall']):.4f}, specificity={float(row['specificity']):.4f}, "
        f"F1={float(row['f1_score']):.4f}, ROC-AUC={float(row['roc_auc']):.4f}, "
        f"validation threshold={float(row['threshold']):.4f}"
    )
print("Split manifest SHA-256:", MANIFEST_SHA256)
print("Transfer checkpoints:", *(str(path.resolve()) for path in model_paths.values()), sep="\n")
print("Results:", RESULTS_DIR.resolve())
